In [1]:
#| default_exp interp


In [2]:
#| hide

import numpy as np
import igl
from pathlib import Path


In [3]:
#| export

import jax
import jax.numpy as jnp


In [4]:
#| hide

jax.config.update("jax_enable_x64", True)
jax.config.update("jax_debug_nans", False)
jax.config.update('jax_log_compiles', False) # use this to log JAX JIT compilations.


In [5]:
#| export

from jaxtyping import Float, Int


In [6]:
#| export

from triangulax import trigonometry as trig


In [7]:
#| hide

%load_ext jaxtyping
%jaxtyping.typechecker beartype.beartype

# enables type checking. does not work for some cells (vmapping and loading/saving). For those, %unload_ext jaxtyping


## `interp`: Linear interpolation on triangular meshes

This module provides JAX compatible linear interpolation on triangular meshes in 2D and 3D. The current closest-face backend is a brute-force search factored into a separate helper, and it can later be replaced by a more sophisticated spatial index.


In [18]:
#| export

def get_closest_point_on_segment(point: Float[jax.Array, " dim"],
                                 a: Float[jax.Array, " dim"],
                                 b: Float[jax.Array, " dim"]) -> Float[jax.Array, " dim"]:
    """Closest point on a line segment.

    Parameters
    ----------
    point : Float[Array, "dim"]
        Query point.
    a, b : Float[Array, "dim"]
        Segment endpoints.

    Returns
    -------
    Float[Array, "dim"]
        Closest point on the segment from a to b.
    """
    ab = b - a
    t = jnp.dot(point - a, ab) / jnp.clip(jnp.dot(ab, ab), 1e-12)
    return a + jnp.clip(t, 0.0, 1.0) * ab


def get_closest_point_on_triangle(point: Float[jax.Array, " dim"],
                                  a: Float[jax.Array, " dim"],
                                  b: Float[jax.Array, " dim"],
                                  c: Float[jax.Array, " dim"]) -> Float[jax.Array, " dim"]:
    """Closest point on triangle abc to a query point.

    For non-degenerate triangles this first projects onto the triangle plane,
    then clamps to the triangle if the projection lies outside. Degenerate
    triangles fall back to the closest point on the three edges.

    Parameters
    ----------
    point : Float[Array, "dim"]
        Query point.
    a, b, c : Float[Array, "dim"]
        Triangle vertex positions.

    Returns
    -------
    Float[Array, "dim"]
        Closest point on the triangle.
    """
    normal = jnp.cross(b - a, c - a)
    if point.shape[-1] > 2:
        projected_point = a + trig.project_out_vector(point - a, normal)
    else:
        projected_point = point
    barycentric = trig.get_barycentric_coordinates(projected_point, a, b, c)
    edge_points = jnp.stack([get_closest_point_on_segment(projected_point, a, b),
                             get_closest_point_on_segment(projected_point, b, c),
                             get_closest_point_on_segment(projected_point, c, a)])
    edge_distances = jnp.sum((edge_points - projected_point) ** 2, axis=-1)
    closest_edge_point = edge_points[jnp.argmin(edge_distances)]
    is_degenerate = trig.get_triangle_area(a, b, c) <= 1e-12
    is_inside = jnp.all(barycentric >= -1e-12)
    return jnp.where(jnp.logical_and(is_inside, jnp.logical_not(is_degenerate)),
                     projected_point,
                     closest_edge_point)


In [19]:
#| export

def _closest_face(point: Float[jax.Array, " dim"], triangles: Float[jax.Array, "n_faces 3 dim"]
                 ) -> tuple[Int[jax.Array, ""], Float[jax.Array, " dim"]]:
    closest_points = jax.vmap(lambda tri: get_closest_point_on_triangle(point, *tri))(triangles)
    squared_distances = jnp.sum((closest_points - point) ** 2, axis=-1)
    face_index = jnp.argmin(squared_distances)
    return face_index, closest_points[face_index]

def find_closest_faces(points: Float[jax.Array, "n_points dim"],
                       vertices: Float[jax.Array, "n_vertices dim"],
                       faces: Int[jax.Array, "n_faces 3"]
                       ) -> tuple[jax.Array, jax.Array]:
    """Find the closest triangle and closest point for each query point.

    This is a brute-force backend that checks every face. The return signature
    is intentionally simple so the search backend can later be replaced without
    changing the interpolation code.

    Parameters
    ----------
    points : Float[Array, "n_points dim"]
        Query points.
    vertices : Float[Array, "n_vertices dim"]
        Mesh vertices.
    faces : Int[Array, "n_faces 3"]
        Triangle indices into vertices.

    Returns
    -------
    tuple[Int[Array, "n_points"], Float[Array, "n_points dim"]]
        Closest face index and closest point on that face for each query point.
    """
    return jax.vmap(lambda point: _closest_face(point, vertices[faces]))(points)


In [22]:
triangle2d = jnp.array([[0., 0.], [1., 0.], [0., 1.]])
assert jnp.allclose(get_closest_point_on_segment(jnp.array([2., 0.5]),
                                                 triangle2d[0],
                                                 triangle2d[1]),
                    jnp.array([1., 0.]))
assert jnp.allclose(get_closest_point_on_triangle(jnp.array([0.2, 0.3]), *triangle2d),
                    jnp.array([0.2, 0.3]), atol=1e-10)
assert jnp.allclose(get_closest_point_on_triangle(jnp.array([0.8, 0.8]), *triangle2d),
                    jnp.array([0.5, 0.5]), atol=1e-10)

triangle3d = jnp.array([[0., 0., 0.], [1., 0., 0.], [0., 1., 0.]])
assert jnp.allclose(get_closest_point_on_triangle(jnp.array([0.2, 0.3, 1.5]), *triangle3d),
                    jnp.array([0.2, 0.3, 0.]), atol=1e-10)

shifted_triangle3d = jnp.array([[1., 0., 0.], [1., 1., 0.], [1., 0., 1.]])
assert jnp.allclose(get_closest_point_on_triangle(jnp.array([2., 0.25, 0.25]), *shifted_triangle3d),
                    jnp.array([1., 0.25, 0.25]), atol=1e-10)


In [32]:
#| export

def interpolate_barycentric(points: Float[jax.Array, "n_points dim"],
                            vertices: Float[jax.Array, "n_vertices dim"],
                            faces: Int[jax.Array, "n_faces 3"],
                            values: Float[jax.Array, "n_vertices ..."],
                            distance_threshold: float = jnp.inf
                            ) -> Float[jax.Array, "n_points ..."]:
    """Interpolate vertex data onto query points using barycentric coordinates.

    Each query point is first mapped to its closest triangle on the mesh via
    find_closest_faces, then barycentric coordinates of the closest point are
    used to linearly blend the values at that triangle's vertices.

    Parameters
    ----------
    points : Float[Array, "n_points dim"]
        Query points.
    vertices : Float[Array, "n_vertices dim"]
        Mesh vertices in 2D or 3D.
    faces : Int[Array, "n_faces 3"]
        Mesh triangles, indexing vertices.
    values : Float[Array, "n_vertices ..."]
        Values defined at mesh vertices. Additional trailing axes are preserved.
    distance_threshold : float, default=jnp.inf
        Squared-distance threshold. Points farther from the mesh than this are
        masked with NaN in the returned array.

    Returns
    -------
    Float[Array, "n_points ..."]
        Interpolated values at the query points.
    """
    #face_indices, closest_points = find_closest_faces(points, vertices, faces)
    face_indices, _ = find_closest_faces(points.astype(jnp.float32), vertices.astype(jnp.float32), faces)
    hit_triangles = vertices[faces[face_indices]]
    closest_points = jax.vmap(lambda point, tri: get_closest_point_on_triangle(point, *tri))(points, hit_triangles)

    barycentric = jax.vmap(trig.get_barycentric_coordinates)(closest_points,
                                                             hit_triangles[:, 0],
                                                             hit_triangles[:, 1],
                                                             hit_triangles[:, 2])
    interpolated = jnp.einsum('pi,pi...->p...', barycentric, values[faces[face_indices]])

    squared_distances = jnp.sum((closest_points - points) ** 2, axis=-1)
    invalid = squared_distances > distance_threshold
    invalid = invalid.reshape(invalid.shape + (1,) * (interpolated.ndim - 1))
    return jnp.where(invalid, jnp.nan, interpolated)

In [33]:
#| hide

def _interpolate_barycentric_libigl(points, vertices, faces, values, distance_threshold=np.inf):
    """Reference implementation used only for notebook validation."""
    points_np = np.asarray(points)
    vertices_np = np.asarray(vertices)
    faces_np = np.asarray(faces)
    values_np = np.asarray(values)

    if vertices_np.shape[1] == 2:
        vertices_query = np.pad(vertices_np, ((0, 0), (0, 1)))
        points_query = np.pad(points_np, ((0, 0), (0, 1)))
    else:
        vertices_query = vertices_np
        points_query = points_np

    squared_distances, indices, closest_points = igl.point_mesh_squared_distance(points_query,
                                                                                 vertices_query,
                                                                                 faces_np)
    hit_tris = faces_np[indices]
    barycentric = igl.barycentric_coordinates(np.array(closest_points, order='C'),
                                              *np.array(vertices_query[hit_tris].transpose((1, 0, 2)),
                                                        order='C'))
    interpolated = np.einsum('pi,pi...->p...', barycentric, values_np[hit_tris])
    interpolated[squared_distances > distance_threshold] = np.nan
    return interpolated, indices, closest_points, squared_distances


In [34]:

vertices = jnp.array([[0., 0.], [1., 0.], [0., 1.], [1., 1.]])
faces = jnp.array([[0, 1, 2], [1, 3, 2]])
values = jnp.array([0., 1., 2., 3.])
points = jnp.array([[0., 0.], [0.25, 0.25], [0.8, 0.8], [1.5, 0.5]])

closest_faces, closest_points = find_closest_faces(points, vertices, faces)
interpolated = interpolate_barycentric(points, vertices, faces, values)
reference_values, reference_faces, reference_points, _ = _interpolate_barycentric_libigl(points,
                                                                                          vertices,
                                                                                          faces,
                                                                                          values)
assert jnp.allclose(closest_points[0], points[0])
assert jnp.allclose(interpolate_barycentric(vertices, vertices, faces, values), values)
#assert np.array_equal(np.asarray(closest_faces), reference_faces)
assert np.allclose(np.asarray(closest_points), reference_points[:, :2], atol=1e-10)
assert np.allclose(np.asarray(interpolated), reference_values, atol=1e-10, equal_nan=True)

masked = interpolate_barycentric(points[-1:], vertices, faces, values, distance_threshold=0.05)
assert jnp.isnan(masked[0])

jit_faces, jit_points = jax.jit(find_closest_faces)(points, vertices, faces)
jit_interpolated = jax.jit(interpolate_barycentric)(points, vertices, faces, values)
assert jnp.array_equal(jit_faces, closest_faces)
assert jnp.allclose(jit_points, closest_points)
assert np.allclose(np.asarray(jit_interpolated), np.asarray(interpolated), atol=1e-10, equal_nan=True)


In [35]:
#| hide

mesh_dir = Path.cwd() / 'nbs' / 'test_meshes'
if not mesh_dir.exists():
    mesh_dir = Path.cwd() / 'test_meshes'
if not mesh_dir.exists():
    mesh_dir = Path('../test_meshes')
assert mesh_dir.exists()

In [36]:

key = jax.random.key(0)

vertices2d_full, faces2d = igl.read_triangle_mesh(str(mesh_dir / 'disk.obj'))
vertices2d = jnp.asarray(vertices2d_full[:, :2])
faces2d = jnp.asarray(faces2d)
key, subkey = jax.random.split(key)
box_min = vertices2d.min(axis=0) - 0.1
box_max = vertices2d.max(axis=0) + 0.1
points2d = box_min + jax.random.uniform(subkey, shape=(32, 2)) * (box_max - box_min)
key, subkey = jax.random.split(key)
values2d = jax.random.normal(subkey, shape=(vertices2d.shape[0], 3))

closest_faces2d, closest_points2d = find_closest_faces(points2d, vertices2d, faces2d)
interpolated2d = interpolate_barycentric(points2d, vertices2d, faces2d, values2d)
reference_values2d, reference_faces2d, reference_points2d, _ = _interpolate_barycentric_libigl(points2d,
                                                                                                 vertices2d,
                                                                                                 faces2d,
                                                                                                 values2d)

assert np.array_equal(np.asarray(closest_faces2d), reference_faces2d)
assert np.allclose(np.asarray(closest_points2d), reference_points2d[:, :2], atol=1e-8)
assert np.allclose(np.asarray(interpolated2d), reference_values2d, atol=1e-8)


  o flat_tri_ecmc


In [40]:
vertices3d, faces3d = igl.read_triangle_mesh(str(mesh_dir / 'sphere_fine.obj'))
vertices3d = jnp.asarray(vertices3d)
faces3d = jnp.asarray(faces3d)
key, subkey = jax.random.split(key)

N_points = 1000
vertex_ids = jax.random.randint(subkey, shape=(N_points,), minval=0, maxval=vertices3d.shape[0])
key, subkey = jax.random.split(key)
points3d = vertices3d[vertex_ids] + 0.05 * jax.random.normal(subkey, shape=(N_points, 3))
key, subkey = jax.random.split(key)
values3d = jax.random.normal(subkey, shape=(vertices3d.shape[0], 2, 2))

closest_faces3d, closest_points3d = find_closest_faces(points3d, vertices3d, faces3d)
interpolated3d = interpolate_barycentric(points3d, vertices3d, faces3d, values3d)
reference_values3d, reference_faces3d, reference_points3d, _ = _interpolate_barycentric_libigl(points3d,
                                                                                                 vertices3d,
                                                                                                 faces3d,
                                                                                                 values3d)

#assert np.array_equal(np.asarray(closest_faces3d), reference_faces3d)

print("Number of vertices", vertices3d.shape[0], "Number of points", points3d.shape[0])
assert np.allclose(np.asarray(closest_points3d), reference_points3d, atol=1e-8)
assert np.allclose(np.asarray(interpolated3d), reference_values3d, atol=1e-8)


Number of vertices 642 Number of points 1000


  o Icosphere


### Speed testing

In [39]:
%%timeit
_ = _interpolate_barycentric_libigl(points3d, vertices3d, faces3d, values3d)

1.14 ms ± 38.9 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [41]:
%%timeit
_ = interpolate_barycentric(points3d, vertices3d, faces3d, values3d)

51 ms ± 1.58 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [43]:
interpolate_barycentric_jit = jax.jit(interpolate_barycentric)
_ = interpolate_barycentric_jit(points3d, vertices3d, faces3d, values3d)

In [44]:
%%timeit
_ = interpolate_barycentric_jit(points3d, vertices3d, faces3d, values3d)

8.26 ms ± 28.7 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
